# Practice · From question to a figure you can defend

**Diagnose → simple data, simple figures (Practice A) → realistic data: test, check, visualize (Practice B) → defend and present**

*Genomic Data Processing · Day 5 (10/27) · Data visualization in genetics and genomics · about 90 minutes, in pairs*

> **All data in this practice are simulated (computer-generated). No real patient data are used.**
> The simulations use real anchors: real human gene names and hg38 coordinates (MANE Select), and published biology
> such as genes that respond to the steroid drug dexamethasone. So the figures look like real research figures, and
> every realistic dataset comes with a **truth file**: you can test whether a method finds the answer that was planted.

**How to use this notebook.** Run the cells from top to bottom (Shift + Enter). Read the short text before each cell:
it says why the step exists and which new words appear.

* `# CHECK ·` cells compare your number with a reference value and print **✓** or **✗**. If you see ✗, stop and find
  out why before you change anything else.
* **What changed?** says what the data or the figure look like after the step, and which number proves it.
* **Your turn** is a small task: change one thing and look again. The answers are folded (click *Answer*).
* Practice B has two core exercises for everyone (B1, B2). After B2, each pair opens **one elective notebook** in
  `electives/` (B3 GWAS, B4 single cell, B5 copy number, B6 variant callers, B7 classifiers, B8 TP53 lollipop).

| Part | Steps | Minutes |
|---|---|---|
| Set-up | Step 0 | 5 |
| Exercise 1 · Diagnose | Step 1 | 8 |
| Practice A · simple data, simple figures | A1–A5 | 20 |
| Practice B · core | B1 (two groups), B2 (RNA-seq) | 27 |
| Practice B · elective | one of B3–B8 (own notebook) | 20 |
| Defend and present | D1–D3, then one figure per pair | 10 + presentations |

## Step 0 · Set-up  *(5 min)*

**Google Colab:** upload `dataviz_practice_dataset.zip`, then run in a cell `!unzip -q dataviz_practice_dataset.zip`
and `%cd dataviz_practice`, and install the two extra packages with `!pip install -q umap-learn adjustText`.
**Your laptop (Jupyter):** unzip the folder, open a terminal in it, run `pip install -r requirements.txt`, then open
this notebook.

The cell below loads the packages and `dviz.py`, a small file of helpers written for this course (figure style,
colours, and the CHECK function). Open `dviz.py` in a text editor at any time: every helper is a few readable lines.

In [ ]:
# Step 0 · Set-up: imports, folders and figure style
import os, sys, time
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "dviz.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault("NUMBA_CACHE_DIR", str(ROOT / ".numba_cache"))   # needed by umap-learn on some systems
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import stats
import sklearn
import dviz
font = dviz.use_style()
DATA, FIG = ROOT / "data", ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
T_START = time.time()
print(f"python {sys.version.split()[0]} · numpy {np.__version__} · pandas {pd.__version__} · "
      f"matplotlib {matplotlib.__version__} · scipy {scipy.__version__} · scikit-learn {sklearn.__version__}")
print("figure font:", font, "· data folder:", DATA)

In [ ]:
# CHECK · every input file of this notebook is present
needed = ["anscombe.tsv", "mapping_rates.tsv", "expression_by_group.tsv", "timecourse.tsv",
          "celltype_composition.tsv", "methylation_cpg.tsv.gz", "rnaseq_counts.tsv.gz", "rnaseq_metadata.tsv",
          "de_results.tsv", "truth/rnaseq_truth.tsv"]
missing = [f for f in needed if not (DATA / f).exists()]
dviz.check("S0.files_missing", len(missing), 0, note=", ".join(missing))

## Step 1 · Exercise 1 · Diagnose: spot the misleader  *(8 min)*

**Why this step.** Before we make figures, we practise *reading* them critically. The eight figures below were made
from the same data you will use today, and each one breaks at least one principle from Part 2 of the lecture.

**Your turn (in pairs).** For each figure M1–M8 (M for *misleader*) write down (1) the principle it breaks and (2) the fix, in one line
each. Then, if you use an AI assistant, give it **one** figure together with the critique prompt in
`prompts/critique_prompt.md`: what did it find that you missed, and did it invent anything?

In [ ]:
# Step 1 · show the eight misleading figures from bad_figures/
from matplotlib.image import imread
fig, axs = plt.subplots(2, 4, figsize=(13, 6.2))
for k, ax in enumerate(axs.flat, start=1):
    p = ROOT / "bad_figures" / f"M{k}.png"
    if p.exists():
        ax.imshow(imread(p))
    ax.set_title(f"M{k}", loc="left")
    ax.axis("off")
fig.tight_layout()
plt.show()

| Figure | Principle it breaks | Fix |
|---|---|---|
| M1 | | |
| M2 | | |
| M3 | | |
| M4 | | |
| M5 | | |
| M6 | | |
| M7 | | |
| M8 | | |

<details><summary>Answer</summary>

M1 bar of means with SE hides n = 4 and an outlier (show every point, name the test) · M2 bars do not start at zero
(start at 0, or use dots with a labelled zoomed axis) · M3 rainbow colours, no colour bar, rows not scaled (z-score,
diverging map centred at 0, colour bar) · M4 zero-expressing cells drawn on top (draw high values last) · M5 pie with
many slices, 3-D shadow and a legend (sorted bars with direct labels) · M6 red-green, raw p, no thresholds, untested
genes shown (adjusted p, thresholds, colour-blind-safe colours, counts) · M7 a claim based on distances between UMAP
clusters (UMAP keeps neighbours, not distances) · M8 two y-axes make two series look related (both are log2 FC vs 0 h: one shared axis, direct
labels). The full answer key is in `solutions/ANSWER_KEY_bad_figures.md`.
</details>

# Practice A · Simple data, simple figures

Practice A uses small tables that you can read by eye. Each step makes one basic figure and asks you to **change one
thing and look**: a setting, an axis or a colour. The goal is to see how a single choice changes what a reader
concludes.

## Step A1 · Plot the data before you trust a summary  *(4 min)*

**Why this step.** Summary numbers such as the mean or the correlation are useful, but very different data can give
the same numbers. Anscombe built four small datasets in 1973 to show this. These are the real published values.

**New terms**
* **Correlation r** – how close the points are to a straight line (−1 to 1).
* **Linear regression** – the straight line that best predicts y from x (`y = intercept + slope × x`).

In [ ]:
# Step A1a · the same summary numbers for four different datasets
ans = pd.read_csv(DATA / "anscombe.tsv", sep="\t")
rows = []
for name, d in ans.groupby("dataset"):
    slope, intercept = np.polyfit(d["x"], d["y"], 1)
    rows.append({"dataset": name, "n": len(d), "mean y": d["y"].mean(), "variance y": d["y"].var(),
                 "r": np.corrcoef(d["x"], d["y"])[0, 1], "intercept": intercept, "slope": slope})
summary_a1 = pd.DataFrame(rows).round(2)
summary_a1

In [ ]:
# Step A1b · now plot them
fig, axs = plt.subplots(1, 4, figsize=(12, 3), sharex=True, sharey=True)
xs = np.array([2, 20])
for ax, (name, d) in zip(axs, ans.groupby("dataset")):
    ax.scatter(d["x"], d["y"], s=22, color=dviz.BLUE, zorder=3)
    ax.plot(xs, 3 + 0.5 * xs, color=dviz.SLATE, lw=1)                # the shared regression line
    ax.set_title(f"Set {name} · r = {np.corrcoef(d['x'], d['y'])[0, 1]:.2f}")
    ax.set_xlabel("x")
axs[0].set_ylabel("y")
fig.tight_layout()
dviz.savefig(fig, "A1_anscombe")
plt.show()

In [ ]:
# CHECK · all four sets have r = 0.82 and the line y = 3 + 0.5 x
for _, r in summary_a1.iterrows():
    dviz.check(f"A.A1_r_{r['dataset']}", round(r["r"], 2), 0.82, tol=0.005)
dviz.check("A.A1_slope_range", round(summary_a1["slope"].max() - summary_a1["slope"].min(), 3), 0.0, tol=0.002)

**What changed?** The table says the four sets are the same (r = 0.82, y = 3 + 0.5x for all four). The plots say they
are not: Set I is a noisy line, II is a curve, III is a line with one outlier, and IV has no relationship except
one point far to the right.

**Your turn.** Which set would you trust a correlation for? Remove the outlier in Set III
(`d[d["y"] < 12]`) and compute r again.

<details><summary>Answer</summary>

Only Set I. Without its outlier, Set III is an almost perfect line (r ≈ 1.00), so one point lowered r to 0.82.
Rule: plot the data before you report a summary statistic.
</details>

## Step A2 · One gene, two groups: how to show a distribution  *(5 min)*

**Why this step.** A typical first question is "is gene X higher in patients (Case) than in healthy people
(Control)?". The table `expression_by_group.tsv` has the expression of one gene, in TPM (transcripts per million),
for two studies: a small one (4 + 4 people) and a larger one (40 + 40 people).

**New terms**
* **Histogram** – counts of values in bins; the number of bins is a choice.
* **Kernel density estimate (KDE)** – a smooth curve made by adding a small bump for every value; the *bandwidth*
  sets how smooth it is.
* **Box plot** – median (line), quartiles (box, the middle 50%) and whiskers to 1.5 × the box height.
* **Violin** – a mirrored KDE; **strip plot** – every value as a point.

In [ ]:
# Step A2a · the Case group of the larger study, three histograms and three smoothing bandwidths
ebg = pd.read_csv(DATA / "expression_by_group.tsv", sep="\t")
big = ebg[ebg["dataset"] == "large_n"]
case = big.loc[big["group"] == "Case", "expression"].to_numpy()
grid = np.linspace(30, 140, 300)
fig, axs = plt.subplots(1, 4, figsize=(13, 2.8), sharex=True)
for ax, bins in zip(axs[:3], [3, 12, 60]):
    ax.hist(case, bins=bins, range=(40, 130), color=dviz.SKY, edgecolor="white")
    ax.set_title(f"{bins} bins")
for bw, col in [(1.2, dviz.SLATE), (0.4, dviz.BLUE), (0.15, dviz.ORANGE)]:
    axs[3].plot(grid, stats.gaussian_kde(case, bw_method=bw)(grid), color=col, label=f"bandwidth {bw}")
axs[3].set_title("Kernel density")
axs[3].legend(fontsize=7)
for ax in axs:
    ax.set_xlabel("Expression (TPM)")
axs[0].set_ylabel("Cases (n = 40)")
fig.tight_layout()
dviz.savefig(fig, "A2_bins_bandwidth")
plt.show()

In [ ]:
# Step A2b · box, violin, strip and all three together (larger study)
groups = ["Control", "Case"]
vals = [big.loc[big["group"] == g, "expression"].to_numpy() for g in groups]
cols = [dviz.SLATE, dviz.BLUE]
fig, axs = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
axs[0].boxplot(vals, positions=[0, 1], widths=0.5)
axs[1].violinplot(vals, positions=[0, 1], showmedians=True)
for i, (v, c) in enumerate(zip(vals, cols)):
    axs[2].scatter(i + dviz.jitter(len(v), 0.12, seed=i), v, s=10, color=c)
    axs[3].violinplot([v], positions=[i], showextrema=False)
    axs[3].boxplot([v], positions=[i], widths=0.12, showfliers=False)
    axs[3].scatter(i + 0.3 + dviz.jitter(len(v), 0.05, seed=i), v, s=8, color=c)
for ax, t in zip(axs, ["Box", "Violin", "Strip (every point)", "Violin + box + points"]):
    ax.set_title(t)
    ax.set_xticks([0, 1], [f"{g}\nn = {len(v)}" for g, v in zip(groups, vals)])
axs[0].set_ylabel("Expression (TPM)")
fig.tight_layout()
dviz.savefig(fig, "A2_box_violin_points")
plt.show()

In [ ]:
# CHECK · the numbers behind the box plots (medians and quartiles)
for g, v in zip(groups, vals):
    q1, med, q3 = np.percentile(v, [25, 50, 75])
    print(f"{g:8s} n = {len(v)}  median {med:.1f}  quartiles {q1:.1f}-{q3:.1f}  range {v.min():.1f}-{v.max():.1f}")
dviz.check("A.A2_case_median", round(float(np.median(vals[1])), 2))
dviz.check("A.A2_control_median", round(float(np.median(vals[0])), 2))
dviz.check("A.A2_case_above_90", int((vals[1] > 90).sum()), note="cases in the upper subgroup (> 90 TPM)")

**What changed?** With 3 bins, or a wide bandwidth, the cases look like one broad hump. With about 12 bins, or a
bandwidth around 0.4, two groups of cases appear: most cases look like controls, and a subgroup is much higher.
60 bins (or bandwidth 0.15) show mostly noise. The box plot hides the two groups (it shows only five numbers); the
strip plot and the combined plot show them.

**Your turn.** Change `bins=bins` to `bins=6` for the first histogram. Can you still see two groups? What would you
write in a figure legend so that a reader can judge your choice?

<details><summary>Answer</summary>

With 6 bins the two groups are barely visible. Write the number of bins (or the KDE bandwidth) in the legend, and
show the points when n is small enough.
</details>

## Step A3 · Bars start at zero; dots may zoom  *(3 min)*

**Why this step.** `mapping_rates.tsv` has the percentage of sequencing reads that mapped to the genome for three
versions (A, B, C) of a pipeline, on the same four libraries. A bar is read by its **length**, so its axis must start
at zero. A dot is read by its **position**, so a zoomed axis is fine if it is labelled.

In [ ]:
# Step A3 · the same three means drawn three ways
mr = pd.read_csv(DATA / "mapping_rates.tsv", sep="\t")
means = mr.groupby("pipeline")["pct_mapped"].mean()
colors = [dviz.GREY, dviz.GREY, dviz.BLUE]
fig, axs = plt.subplots(1, 3, figsize=(11, 3))
for ax, lo, title in [(axs[0], 95.5, "Bars, axis from 95.5% (wrong)"), (axs[1], 0, "Bars, axis from 0")]:
    ax.bar(means.index, means.values, color=colors, width=0.6)
    ax.set_ylim(lo, 100)
    ax.set_title(title)
    for i, v in enumerate(means.values):
        ax.text(i, v + 0.05 * (100 - lo) / 4.5, f"{v:.1f}", ha="center", va="bottom", fontsize=8)
for i, (p, d) in enumerate(mr.groupby("pipeline")):
    axs[2].scatter(np.full(len(d), i) + dviz.jitter(len(d), 0.06, seed=i), d["pct_mapped"], s=18, color=colors[i] if i == 2 else dviz.SLATE)
    axs[2].hlines(d["pct_mapped"].mean(), i - 0.2, i + 0.2, color=dviz.INK, lw=1.5)
axs[2].set_xticks([0, 1, 2], list(means.index))
axs[2].set_ylim(95.5, 98.5)
axs[2].set_title("Dots + mean, zoomed axis (labelled)")
for ax in axs:
    ax.set_ylabel("Reads mapped (%)")
    ax.set_xlabel("Pipeline version (4 libraries each)")
fig.tight_layout()
dviz.savefig(fig, "A3_bars_from_zero")
plt.show()

In [ ]:
# CHECK · how much bigger does C look than A when the axis starts at 95.5%?
looks = (means["C"] - 95.5) / (means["A"] - 95.5)
true = means["C"] / means["A"]
print(f"means: A {means['A']:.1f} · B {means['B']:.1f} · C {means['C']:.1f}")
dviz.check("A.A3_bar_ratio_looks", round(float(looks), 1), note="bar C height / bar A height with the axis at 95.5%")
dviz.check("A.A3_true_ratio", round(float(true), 3), 1.02, tol=0.001)

**What changed?** Same numbers, different impressions. From 95.5%, bar C is about four times taller than bar A,
although C maps only 1.9 percentage points (2%) more reads. From zero, the three bars look almost equal, which is
the truth. The dot plot zooms in honestly: the axis is labelled, and you also see that the four libraries agree.

**Your turn.** In the first panel change `95.5` to `96.0`. How many times taller does C look now?

<details><summary>Answer</summary>

(98.0 − 96.0) / (96.1 − 96.0) = 20 times. The closer the baseline is to the smallest bar, the bigger the distortion.
</details>

## Step A4 · Lines for time, bars for composition  *(4 min)*

**Why this step.** Two common tables: (1) a **time course** – expression of three genes at 0, 1, 2, 4, 8 and 24 hours
after adding dexamethasone (a steroid drug) to cells, three replicate wells per time; (2) a **composition** – the
percentage of each blood cell type in four donors (from the single-cell data of elective B4).
A line connects ordered values, so it suits time. Composition is about parts of a whole.

**New terms**
* **log2 fold change (log2 FC)** – log2 of (expression at time t / expression at 0 h). +1 = doubled, −1 = halved.
* **SD (standard deviation)** – the spread of the replicates.

In [ ]:
# Step A4a · time course: mean ± SD of 3 replicates, the replicates as points, direct labels
tc = pd.read_csv(DATA / "timecourse.tsv", sep="\t")
fig, ax = plt.subplots(figsize=(6, 3.3))
for g, col in zip(["TSC22D3", "FKBP5", "PER1"], [dviz.BLUE, dviz.ORANGE, dviz.SLATE]):
    d = tc[tc["gene"] == g]
    m = d.groupby("time_h")["log2fc_vs_0h"].agg(["mean", "std"])
    ax.errorbar(m.index, m["mean"], yerr=m["std"], color=col, marker="o", ms=4, capsize=2, lw=1.6)
    ax.scatter(d["time_h"], d["log2fc_vs_0h"], color=col, s=8, alpha=0.5)
    ax.text(24.6, m["mean"].iloc[-1], g, color=col, va="center", fontsize=9)
ax.axhline(0, color=dviz.LIGHT, lw=1, zorder=0)
ax.set_xticks([0, 1, 2, 4, 8, 24])
ax.set_xlim(-0.5, 28)
ax.set_xlabel("Hours after dexamethasone")
ax.set_ylabel("log2 FC vs 0 h")
ax.set_title("Mean ± SD, n = 3 wells per time")
fig.tight_layout()
dviz.savefig(fig, "A4_timecourse")
plt.show()

In [ ]:
# Step A4b · composition: a pie vs sorted bars (all donors), and per donor
comp = pd.read_csv(DATA / "celltype_composition.tsv", sep="\t")
overall = comp.groupby("cell_type")["n_cells"].sum().sort_values()
pct = 100 * overall / overall.sum()
fig, axs = plt.subplots(1, 3, figsize=(13, 3.4), gridspec_kw={"width_ratios": [1, 1.2, 1.4]})
colour = dict(zip(pct.index[::-1], dviz.OKABE_ITO))         # one colour per cell type, used in both panels
axs[0].pie(pct.values[::-1], colors=[colour[c] for c in pct.index[::-1]], startangle=90, counterclock=False)
axs[0].set_title("Pie: which is bigger?")
axs[1].barh(pct.index, pct.values, color=dviz.SKY)
for i, v in enumerate(pct.values):
    axs[1].text(v + 0.4, i, f"{v:.1f}%", va="center", fontsize=8)
axs[1].set_xlabel("Cells (%)")
axs[1].set_title(f"Sorted bars, n = {overall.sum():,} cells")
wide = comp.pivot(index="donor", columns="cell_type", values="percent")[list(pct.index[::-1])]
left = np.zeros(len(wide))
for ct in wide.columns:
    axs[2].barh(wide.index, wide[ct], left=left, color=colour[ct], label=ct)
    left += wide[ct].to_numpy()
axs[2].set_xlabel("Cells (%)")
axs[2].set_title("Stacked, per donor")
axs[2].legend(ncol=2, fontsize=7, bbox_to_anchor=(1.0, -0.25), loc="upper right")
fig.tight_layout()
dviz.savefig(fig, "A4_composition")
plt.show()

In [ ]:
# CHECK · percentages add up to 100 in every donor; the two cell types the lecture compared
dviz.check("A.A4_donor_sum_max_error", round(float((comp.groupby("donor")["percent"].sum() - 100).abs().max()), 1), 0.0, tol=0.15)
dviz.check("A.A4_cd14_mono_pct", round(float(pct["CD14 Mono"]), 1), 19.0, tol=0.05)
dviz.check("A.A4_memory_cd4_pct", round(float(pct["Memory CD4 T"]), 1), 16.3, tol=0.05)
dviz.check("A.A4_fkbp5_24h", round(float(tc.query("gene == 'FKBP5' and time_h == 24")["log2fc_vs_0h"].mean()), 2))

**What changed?** The time course shows three different shapes: TSC22D3 rises fast and stays up, FKBP5 rises slowly
and is still rising at 24 h, PER1 peaks at 1 h and falls back. Direct labels replace the legend. In the pie you cannot
tell whether CD14 monocytes (19.0%) are more frequent than memory CD4 T cells (16.3%); the sorted bars answer at a
glance. In the stacked bars only the first segment shares a baseline, so the middle cell types are hard to compare
across donors.

**Your turn.** Remove `marker="o"` and the scatter line from A4a. What information is lost?

<details><summary>Answer</summary>

You no longer see when the samples were taken (the uneven spacing 0, 1, 2, 4, 8, 24 h) or how much the replicates
vary. A line between sparse time points also suggests values that were never measured.
</details>

## Step A5 · Many points: transparency and 2-D density  *(4 min)*

**Why this step.** `methylation_cpg.tsv.gz` has 60,000 CpG sites measured in the same sample by two methods:
bisulfite sequencing (WGBS, the reference) and nanopore sequencing. The **β value** is the fraction of reads that are
methylated at a CpG (0 = never, 1 = always). With 60,000 points a normal scatter plot becomes a solid blob.

**New terms**
* **Pearson r** – correlation of the values (sensitive to outliers); **Spearman ρ** – correlation of the ranks.
* **Hexbin** – a 2-D histogram with hexagonal bins; colour = how many points fall in each bin.

In [ ]:
# Step A5 · opaque points, transparent points, and a hexbin density
me = pd.read_csv(DATA / "methylation_cpg.tsv.gz", sep="\t")
x, y = me["beta_wgbs"], me["beta_nanopore"]
r, rho = x.corr(y), x.corr(y, method="spearman")
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
axs[0].scatter(x, y, s=2, color=dviz.BLUE, rasterized=True)
axs[0].set_title(f"{len(me):,} points, opaque")
axs[1].scatter(x, y, s=2, color=dviz.BLUE, alpha=0.04, rasterized=True)
axs[1].set_title("Transparency (alpha 0.04)")
hb = axs[2].hexbin(x, y, gridsize=45, bins="log", cmap="viridis", mincnt=1)
fig.colorbar(hb, ax=axs[2], label="CpGs (log scale)")
axs[2].set_title(f"Hexbin · r = {r:.3f}, ρ = {rho:.3f}")
for ax in axs:
    ax.set_xlabel("Bisulfite (WGBS) β")
axs[0].set_ylabel("Nanopore β")
fig.tight_layout()
dviz.savefig(fig, "A5_methylation_hexbin")
plt.show()

In [ ]:
# CHECK · correlation, and the disagreement at intermediate methylation vs near 0 or 1
mid = me[(x > 0.3) & (x < 0.7)]
ext = me[(x < 0.1) | (x > 0.9)]
d_mid = (mid["beta_wgbs"] - mid["beta_nanopore"]).abs().median()
d_ext = (ext["beta_wgbs"] - ext["beta_nanopore"]).abs().median()
dviz.check("A.A5_pearson_r", round(float(r), 3))
dviz.check("A.A5_spearman_rho", round(float(rho), 3))
dviz.check("A.A5_median_abs_diff_mid", round(float(d_mid), 3), note=f"β 0.3-0.7, n = {len(mid):,}")
dviz.check("A.A5_median_abs_diff_extreme", round(float(d_ext), 3), note=f"β < 0.1 or > 0.9, n = {len(ext):,}")
dviz.check("A.A5_mid_worse_than_extreme", bool(d_mid > 2 * d_ext), True)

**What changed?** The opaque plot hides where most points are. Transparency helps; the hexbin with a log colour scale
shows that most CpGs sit near 0 or 1, and that the two methods disagree more at intermediate methylation (β 0.3–0.7),
where sampling a limited number of reads matters most. The correlation (r ≈ 0.98) summarises the overall trend but not
this. Spearman's ρ is lower because thousands of CpGs are tied near 0 or 1.

**Your turn.** Remove `bins="log"` from the hexbin. What happens to the intermediate region?

<details><summary>Answer</summary>

With a linear colour scale the two dense corners take almost the whole colour range, so the intermediate region
looks empty. A log scale is the right choice when counts span several orders of magnitude; say so on the colour bar.
</details>

# Practice B · Realistic data: test, check, then visualize

In Practice B a figure is the **last** step of an analysis. Every exercise has the same four beats:

1. **Question** – the biological question in one sentence.
2. **Test** – run the statistic or method that a real analysis would use, and print the numbers.
3. **Check** – compare the numbers with the planted truth or a reference value (`# CHECK ·` cells).
4. **Visualize** – make the figure that shows the result honestly: n, the test, thresholds, units.

You may write the plotting code yourself, use the cells provided, or ask an AI assistant with a prompt card from
`prompts/prompt_cards.md`. Whichever you choose, the CHECK cells decide whether the figure is right.

## Step B1 · Two groups: which test, which error bar?  *(7 min)*

### Question
Is the gene from Step A2 higher in Case than in Control, in the small study (4 + 4 people) and in the larger one
(40 + 40)?

**New terms**
* **Welch t-test** – compares the two **means**; assumes roughly normal values; allows different spreads.
* **Mann–Whitney U test** – compares the **ranks** (who is larger than whom); no normality assumption.
* **SD** – spread of the data. **SEM** = SD / √n – how precisely the mean is known.
  **95% CI** ≈ mean ± t × SEM – the range that would contain the true mean in 95% of repeated studies.

### Test

In [ ]:
# Step B1a · two tests on two studies
rows = []
for ds in ["small_n", "large_n"]:
    d = ebg[ebg["dataset"] == ds]
    ctrl = d.loc[d["group"] == "Control", "expression"].to_numpy()
    case = d.loc[d["group"] == "Case", "expression"].to_numpy()
    welch = stats.ttest_ind(case, ctrl, equal_var=False)
    mwu = stats.mannwhitneyu(case, ctrl, alternative="two-sided")       # exact p-value for small n
    rows.append({"study": ds, "n per group": len(ctrl), "median Control": np.median(ctrl),
                 "median Case": np.median(case), "mean Control": ctrl.mean(), "mean Case": case.mean(),
                 "Welch p": welch.pvalue, "Mann-Whitney p": mwu.pvalue})
b1 = pd.DataFrame(rows)
b1.round(4)

### Check

In [ ]:
# CHECK · the two tests disagree for n = 4; both agree for n = 40
s, l = b1.iloc[0], b1.iloc[1]
dviz.check("B1.small_median_control", round(s["median Control"], 1), 60.8, tol=0.05)
dviz.check("B1.small_median_case", round(s["median Case"], 1), 89.8, tol=0.05)
dviz.check("B1.small_mwu_p", round(s["Mann-Whitney p"], 4), round(2 / 70, 4), tol=0.0001,
           note="2/70: the smallest two-sided p possible with 4 + 4")
dviz.check("B1.small_welch_p", round(s["Welch p"], 3), 0.107, tol=0.002)
dviz.check("B1.large_mwu_p", float(f"{l['Mann-Whitney p']:.2g}"), between=(0, 0.001))
dviz.check("B1.large_welch_p", float(f"{l['Welch p']:.2g}"), between=(0, 0.001))

In [ ]:
# Step B1b · three error bars for the same 4 Case values: SD, SEM, 95% CI
v = ebg.query("dataset == 'small_n' and group == 'Case'")["expression"].to_numpy()
n, m, sd = len(v), v.mean(), v.std(ddof=1)
sem = sd / np.sqrt(n)
tcrit = stats.t.ppf(0.975, n - 1)
bars = pd.Series({"SD": sd, "SEM": sem, "95% CI": tcrit * sem}).round(1)
print(f"n = {n}, mean = {m:.1f} TPM, t(0.975, df = {n - 1}) = {tcrit:.2f}")
bars

In [ ]:
# CHECK · the 95% CI is t times wider than the SEM bar (t = 3.18 for n = 4)
dviz.check("B1.ci_over_sem", round(float(bars["95% CI"] / bars["SEM"]), 2), 3.18, tol=0.01)
dviz.check("B1.small_case_sd", round(float(sd), 1))

### Visualize

In [ ]:
# Step B1c · the figure: every point, n under each label, the test named (no stars)
fig, axs = plt.subplots(1, 3, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1, 1, 1.15]})
for ax, ds, title in [(axs[0], "small_n", "Small study"), (axs[1], "large_n", "Larger study")]:
    d = ebg[ebg["dataset"] == ds]
    vv = [d.loc[d["group"] == g, "expression"].to_numpy() for g in ["Control", "Case"]]
    ax.boxplot(vv, positions=[0, 1], widths=0.35, showfliers=False, medianprops={"color": dviz.INK})
    for i, (x, c) in enumerate(zip(vv, [dviz.SLATE, dviz.BLUE])):
        ax.scatter(i + dviz.jitter(len(x), 0.09, seed=i), x, s=14, color=c, zorder=3)
    p = b1.set_index("study").loc[ds, "Mann-Whitney p"]
    ax.set_title(f"{title} · Mann-Whitney p = {p:.2g}")
    ax.set_xticks([0, 1], [f"Control\nn = {len(vv[0])}", f"Case\nn = {len(vv[1])}"])
    ax.set_ylim(0, 160)
    ax.set_ylabel("Expression (TPM)")
labels = ["data", "mean ± SD", "mean ± SEM", "mean ± 95% CI"]
axs[2].scatter(np.zeros(n) + dviz.jitter(n, 0.05), v, color=dviz.GREY, s=16)
for k, (lab, half, c) in enumerate(zip(labels[1:], bars.values, [dviz.ORANGE, dviz.BLUE, dviz.GREEN]), start=1):
    axs[2].errorbar(k, m, yerr=half, fmt="o", color=c, capsize=4, lw=1.6)
    axs[2].text(k + 0.12, m + half, f"±{half:.1f}", ha="left", va="center", fontsize=8, color=c)
axs[2].set_xticks(range(4), labels, fontsize=8)
axs[2].set_title(f"Same 4 Cases, three error bars")
axs[2].set_ylim(0, 200)
fig.tight_layout()
dviz.savefig(fig, "B1_two_groups")
plt.show()

**What changed?** In the small study the Mann–Whitney test gives p = 0.029 (all four cases are above all four
controls), while the Welch t-test gives p = 0.11, because one case at 150 TPM makes the spread large. With n = 4,
0.029 is the smallest p-value the Mann–Whitney test can produce, so neither result is convincing: show the points.
In the larger study both tests give p < 0.001, and the figure shows what the p-value cannot: only a subgroup of cases
is high. The SEM bar is the shortest, but the 95% CI, the honest statement of uncertainty, is 3.2 times wider.

**Your turn.** A figure says "mean ± error, n = 4, *". Which two pieces of information are missing?

<details><summary>Answer</summary>

What the error bar is (SD, SEM or CI) and which test produced the star (and its p-value). Also whether n counts
people (biological replicates) or wells from one sample (technical replicates).
</details>

## Step B2 · Bulk RNA-seq: from counts to a volcano plot you can defend  *(20 min)*

### Question
**Which genes change when airway smooth-muscle cells are treated with dexamethasone?** Dexamethasone is a steroid
(glucocorticoid) drug used for asthma. Six cultures were treated and six were left untreated (controls). The twelve
RNA libraries were sequenced in **two batches** (A and B), with 3 controls and 3 treated samples in each batch.

The data are simulated in the style of a real study (Himes et al., 2014): the gene names are real (8,000 human
protein-coding genes), and the strongest induced genes include known glucocorticoid-response genes such as *FKBP5*,
*TSC22D3* and *PER1*. The file `data/truth/rnaseq_truth.tsv` says which genes were planted as changed: use it only in
the CHECK cells.

### B2.1 · Look at the data first  *(3 min)*

**New terms**
* **Count matrix** – one row per gene, one column per sample; each number is the reads assigned to that gene.
* **Library size** – total reads of a sample. Samples are sequenced to different depths.
* **Batch** – samples processed together (same day, kit or run). Batches can shift the measurements.

In [ ]:
# Step B2.1 · read the counts and the sample table
counts = pd.read_csv(DATA / "rnaseq_counts.tsv.gz", sep="\t", index_col="gene")
meta = pd.read_csv(DATA / "rnaseq_metadata.tsv", sep="\t")
annot = counts[["gene_id", "chrom"]]
counts = counts.drop(columns=["gene_id", "chrom"])[meta["sample"]]     # genes x samples, same order as meta
print(counts.shape, "(genes, samples)")
print(pd.crosstab(meta["condition"], meta["batch"]))
counts.iloc[:5, :6]

In [ ]:
# Step B2.1b · library sizes (QC): do the samples have similar sequencing depth?
libsize = counts.sum(axis=0) / 1e6
fig, ax = plt.subplots(figsize=(8, 2.6))
cols = [dviz.BLUE if c == "treated" else dviz.SLATE for c in meta["condition"]]
ax.bar(meta["sample"], libsize.values, color=cols, width=0.65)
ax.axvline(5.5, color=dviz.GREY, lw=1, ls=":")
ax.text(2.5, libsize.max() * 1.02, "batch A", ha="center", fontsize=8)
ax.text(8.5, libsize.max() * 1.02, "batch B", ha="center", fontsize=8)
ax.set_ylabel("Reads (millions)")
ax.set_title("Library size per sample (grey = control, blue = treated)")
ax.tick_params(axis="x", labelsize=7)
fig.tight_layout()
dviz.savefig(fig, "B2_library_sizes")
plt.show()

In [ ]:
# CHECK · 8,000 genes x 12 samples; balanced design (3 + 3 in each batch)
dviz.check("B2.n_genes", counts.shape[0], 8000)
dviz.check("B2.n_samples", counts.shape[1], 12)
dviz.check("B2.balanced", bool((pd.crosstab(meta["condition"], meta["batch"]) == 3).all().all()), True)
dviz.check("B2.libsize_ratio_max_min", round(float(libsize.max() / libsize.min()), 2), note="deepest / shallowest library")

**What changed?** Nothing yet: we only looked. The deepest library has 2.27 times more reads than the
shallowest, so raw counts cannot be compared between samples. The design is **balanced**: each batch has three
controls and three treated samples, so batch and treatment can be separated.

### B2.2 · Normalise: make the samples comparable  *(3 min)*

**Why this step.** A gene with 200 reads in a deep library may be *less* expressed than a gene with 150 reads in a
shallow one. DESeq2 divides each sample by a **size factor** computed with the *median-of-ratios* method: compare
every gene with its average across samples, and take the median ratio per sample. The median ignores the few genes
that really change.

**New terms**
* **Geometric mean** – the average on the log scale; robust to one very large value.
* **log2(x + 1)** – a log scale on which a difference of 1 means a two-fold change; + 1 avoids log(0).

In [ ]:
# Step B2.2 · median-of-ratios size factors (the DESeq2 method) in four lines
expressed = (counts > 0).all(axis=1)                         # genes with a count in every sample
log_geo_mean = np.log(counts[expressed]).mean(axis=1)        # log of the "average sample"
size_factors = np.exp((np.log(counts[expressed]).sub(log_geo_mean, axis=0)).median(axis=0))
norm = counts / size_factors                                 # normalised counts
log2n = np.log2(norm + 1)
pd.DataFrame({"library size (M)": libsize.round(2), "size factor": size_factors.round(3)}).T

In [ ]:
# CHECK · size factors are centred near 1 and follow the library sizes (closely, not exactly)
dviz.check("B2.sf_geomean", round(float(np.exp(np.log(size_factors).mean())), 3), between=(0.97, 1.03))
dviz.check("B2.sf_vs_libsize_r", round(float(np.corrcoef(size_factors, libsize)[0, 1]), 2), between=(0.85, 1.0))
dviz.check("B2.sf_range", f"{size_factors.min():.2f}-{size_factors.max():.2f}")

**What changed?** Each sample was divided by its size factor (0.65–1.47 here), so a normalised count of 200 now means
the same in every sample. The size factors follow the library sizes closely (r = 0.99) but not exactly: the
median-of-ratios method also corrects for genes that change strongly and take up a larger share of the reads.

### B2.3 · Quality control with PCA: biology or batch?  *(4 min)*

**Why this step.** Before testing any gene, check what makes the samples different overall. **PCA** (principal
component analysis) finds the directions in which the samples differ most. If samples split by batch more than by
treatment, every later result is suspect. We use the 1,000 most variable genes among genes with a mean normalised
count of at least 10 (very low counts are mostly noise).

**New terms**
* **PC1, PC2** – the first and second directions of largest variance; **% variance** says how much of the total
  variation each one captures.

In [ ]:
# Step B2.3 · PCA on the 1,000 most variable genes (after removing low-count genes)
from sklearn.decomposition import PCA
keep = norm.mean(axis=1) >= 10
top = log2n[keep].var(axis=1).nlargest(1000).index
X = log2n.loc[top].T                                         # samples x genes
pca = PCA(n_components=5).fit(X - X.mean())
S = pca.transform(X - X.mean())
vr = pca.explained_variance_ratio_ * 100
fig, ax = plt.subplots(figsize=(4.8, 3.8))
for cond, col in [("control", dviz.SLATE), ("treated", dviz.BLUE)]:
    for b, mk in [("A", "o"), ("B", "^")]:
        msk = ((meta["condition"] == cond) & (meta["batch"] == b)).to_numpy()
        ax.scatter(S[msk, 0], S[msk, 1], color=col, marker=mk, s=60, edgecolor="white", label=f"{cond}, batch {b}")
ax.axhline(0, color=dviz.LIGHT, lw=1, zorder=0)
ax.axvline(0, color=dviz.LIGHT, lw=1, zorder=0)
ax.set_xlabel(f"PC1 ({vr[0]:.0f}% variance)")
ax.set_ylabel(f"PC2 ({vr[1]:.0f}% variance)")
ax.set_title("1,000 most variable genes")
ax.legend(fontsize=7, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.2))
fig.tight_layout()
dviz.savefig(fig, "B2_pca")
plt.show()

In [ ]:
# CHECK · what do PC1 and PC2 correspond to?
treated = (meta["condition"] == "treated").to_numpy().astype(float)
batch_b = (meta["batch"] == "B").to_numpy().astype(float)
dviz.check("B2.pc1_pct", round(float(vr[0]), 1))
dviz.check("B2.pc2_pct", round(float(vr[1]), 1))
dviz.check("B2.pc1_vs_treatment_r", round(abs(float(np.corrcoef(S[:, 0], treated)[0, 1])), 2), between=(0.9, 1.0))
dviz.check("B2.pc2_vs_batch_r", round(abs(float(np.corrcoef(S[:, 1], batch_b)[0, 1])), 2), between=(0.9, 1.0))

**What changed?** Twelve samples with 1,000 genes each became twelve points in two dimensions. PC1 (39% of the
variation) separates treated from control: treatment is the biggest effect, as we hope. PC2 (23%)
separates batch A from batch B. That is a warning, not a result: the batch effect is real and must be in the model.

**Your turn.** Colour by batch and use the marker shape for condition. Which version of the figure would you show in
a paper about the drug, and which in a quality-control report?

<details><summary>Answer</summary>

For the drug paper, colour = condition (the message) and shape = batch (the check). For a QC report, colour = batch,
because the question is "is there a batch effect?". Either way, two variables need two channels, and the shape keeps
working in greyscale.
</details>

### B2.4 · Test every gene, with and without batch in the model  *(5 min)*

**Why this step.** For each gene we fit a small **linear model** on the log2 normalised expression:
`expression = baseline + batch effect + treatment effect + noise`, and test whether the treatment effect is zero.
Without the batch term, the batch differences end up in the "noise" and real changes look less certain.
This is a simplified version of what limma or DESeq2 do; we write it out so that you can see each part.

**New terms**
* **p-value histogram** – if most genes do not change, their p-values are spread evenly between 0 and 1 (a flat
  floor); real changes add a spike near 0.
* **Multiple testing** – with 6,940 tests at p < 0.05, about 347 genes pass by chance alone.
* **Benjamini–Hochberg (BH)** – adjusts p-values so that among the genes you call, the expected fraction of false
  ones (the **false discovery rate, FDR**) is at most 5%.

In [ ]:
# Step B2.4a · one linear model per gene (all genes at once with matrix algebra)
Y = log2n[keep].to_numpy().T                                  # samples x genes
def treatment_p(design):
    beta, *_ = np.linalg.lstsq(design, Y, rcond=None)         # fit every gene at once
    resid = Y - design @ beta
    df = design.shape[0] - design.shape[1]                    # residual degrees of freedom
    se = np.sqrt((resid ** 2).sum(axis=0) / df * np.linalg.inv(design.T @ design)[-1, -1])
    t = beta[-1] / se                                         # the last column is "treated"
    return 2 * stats.t.sf(np.abs(t), df), beta[-1]
one = np.ones(len(meta))
p_with, lfc_with = treatment_p(np.column_stack([one, batch_b, treated]))
p_without, _ = treatment_p(np.column_stack([one, treated]))
print(f"{keep.sum():,} genes tested by hand; raw p < 0.05: with batch {np.sum(p_with < 0.05):,}, "
      f"without batch {np.sum(p_without < 0.05):,}")

In [ ]:
# Step B2.4b · p-value histograms: check that the test behaved before reading any result
fig, axs = plt.subplots(1, 2, figsize=(9, 2.8), sharey=True)
for ax, p, t in [(axs[0], p_with, "Model with batch"), (axs[1], p_without, "Model without batch")]:
    ax.hist(p, bins=40, range=(0, 1), color=dviz.SKY, edgecolor="white")
    ax.axhline(len(p) / 40 * 0.93, color=dviz.SLATE, ls="--", lw=1)
    ax.set_title(t)
    ax.set_xlabel("raw p-value")
axs[0].set_ylabel("Genes")
axs[0].text(0.55, len(p_with) / 40 * 1.25, "flat floor = genes that do not change", fontsize=7, color=dviz.SLATE)
fig.tight_layout()
dviz.savefig(fig, "B2_pvalue_histograms")
plt.show()

In [ ]:
# Step B2.4c · Benjamini-Hochberg, then compare with the truth (allowed in a CHECK)
truth = pd.read_csv(DATA / "truth" / "rnaseq_truth.tsv", sep="\t", index_col="gene")
is_de = truth.loc[log2n.index[keep], "is_de"].to_numpy() == 1
rows = []
for name, p in [("with batch", p_with), ("without batch", p_without)]:
    called = dviz.bh_adjust(p) < 0.05
    rows.append({"model": name, "called (FDR 5%)": int(called.sum()),
                 "truly changed": int((called & is_de).sum()), "not planted": int((called & ~is_de).sum()),
                 "false discovery proportion": round(float((called & ~is_de).sum() / max(called.sum(), 1)), 3)})
b2_tests = pd.DataFrame(rows)
b2_tests

In [ ]:
# CHECK · BH kept the false fraction near 5%; adding batch found more real changes
w, wo = b2_tests.iloc[0], b2_tests.iloc[1]
dviz.check("B2.hand_called_with_batch", int(w["called (FDR 5%)"]))
dviz.check("B2.hand_called_without_batch", int(wo["called (FDR 5%)"]))
dviz.check("B2.hand_fdp_with_batch", float(w["false discovery proportion"]), between=(0.0, 0.08))
dviz.check("B2.hand_true_gain_from_batch", int(w["truly changed"] - wo["truly changed"]), between=(1, 100))
dviz.check("B2.expected_false_if_unadjusted", int(round(keep.sum() * 0.05)), note="tests x 0.05 if nothing changed")
dviz.check("B2.hand_tested", int(keep.sum()))
dviz.check("B2.raw_p05_with_batch", int((p_with < 0.05).sum()))
dviz.check("B2.raw_p05_not_planted", int(((p_with < 0.05) & ~is_de).sum()))

**What changed?** We now have one p-value per gene. Both histograms have a flat floor and a spike at 0, so the
tests behaved. After BH, about 5% of the called genes were not planted, as BH promises. With batch in the model we
find more truly changed genes, because the batch differences no longer inflate the noise. In a balanced design
ignoring batch costs **power**. In an unbalanced design, for example all treated samples in one batch, it would also
create **false** discoveries, and no model could rescue it.

**Your turn.** How many genes have a raw p < 0.05 in the model with batch, and how many of those were not planted?
Compare with `keep.sum() * 0.05`.

<details><summary>Answer</summary>

`np.sum(p_with < 0.05)` is 683, of which 223 were not planted, close to the 347 expected by chance
(6,940 tests × 0.05). That is why we adjust: with raw p < 0.05, 33% of the "hits" would be false.
</details>

### B2.5 · What the standard tool says: DESeq2  *(4 min)*

**Why this step.** Counts are not normally distributed: low counts are noisy, and the spread grows with the mean.
**DESeq2** models counts with a *negative binomial* distribution, normalises with the size factors from B2.2, borrows
information across genes to estimate their variability, and **shrinks** the fold changes of uncertain genes toward 0.
The instructor ran PyDESeq2 (the Python version of DESeq2) with the design `~ batch + condition`; the result is
`data/de_results.tsv`. The optional cell below re-runs it if `pydeseq2` is installed (about 20 s).

**New terms**
* **log2FC_shrunk** – fold change after shrinkage (apeGLM): reliable genes keep their value, noisy ones move to 0.
* **padj = NA** – the gene was **not tested** (all zeros, too few reads, or an outlier). Never plot NA as 0 or 1.

In [ ]:
# Step B2.5a · read the DESeq2 results and count up / down genes with BOTH thresholds
de = pd.read_csv(DATA / "de_results.tsv", sep="\t")
tested = de["padj"].notna()
up = tested & (de["padj"] < 0.05) & (de["log2FC_shrunk"] > 1)
dn = tested & (de["padj"] < 0.05) & (de["log2FC_shrunk"] < -1)
fdr_only = tested & (de["padj"] < 0.05)
de0 = pd.read_csv(DATA / "de_results_no_batch.tsv", sep="\t")
print(f"tested {tested.sum():,} · not tested (padj NA) {(~tested).sum():,}")
print(f"FDR < 5%: {fdr_only.sum():,} genes with batch in the design, {(de0['padj'] < 0.05).sum():,} without")
print(f"FDR < 5% and |log2FC_shrunk| > 1: {up.sum():,} up, {dn.sum():,} down")

In [ ]:
# Step B2.5b (optional) · re-run DESeq2 yourself (needs: pip install pydeseq2)
try:
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats
    RERUN = False                    # set to True to re-run (about 20 s)
    if RERUN:
        dds = DeseqDataSet(counts=counts.T, metadata=meta.set_index("sample")[["condition", "batch"]],
                           design="~batch + condition", quiet=True)
        dds.deseq2()
        ds = DeseqStats(dds, contrast=["condition", "treated", "control"], quiet=True)
        ds.summary()
        print((ds.results_df["padj"] < 0.05).sum(), "genes at FDR < 5% in your run")
except ImportError:
    print("pydeseq2 is not installed: the provided de_results.tsv is used (that is fine).")

In [ ]:
# CHECK · counts, and how many highlighted genes were really planted
planted = truth.loc[de["gene"], "is_de"].to_numpy() == 1
dviz.check("B2.tested", int(tested.sum()), 7392)
dviz.check("B2.untested", int((~tested).sum()), 608)
dviz.check("B2.up", int(up.sum()), 177)
dviz.check("B2.down", int(dn.sum()), 106)
dviz.check("B2.fdr05_with_batch", int(fdr_only.sum()), 396)
dviz.check("B2.fdr05_without_batch", int((de0["padj"] < 0.05).sum()), 367)
dviz.check("B2.volcano_genes_not_planted", int(((up | dn) & ~planted).sum()), 0, tol=3)
dviz.check("B2.fdr_only_not_planted_pct", round(100 * float((fdr_only & ~planted).sum() / fdr_only.sum()), 1),
           note="% of FDR-only genes that were not planted")
dviz.check("B2.any_gene_up_and_down", bool((up & dn).any()), False)

**What changed?** DESeq2 tested 7,392 genes; 608 were not tested (padj NA) and must be left out of the plot and
counted in the legend. With batch in the design DESeq2 calls 396 genes at FDR < 5%, against 367 without batch.
With **both** thresholds (FDR < 5% and |log2FC| > 1) there are 177 up and 106 down, and 99.6% of them were planted.
Among the FDR-only genes, 5.8% were not planted: most have small fold changes in highly expressed genes, where a
tiny error in the estimated variability is enough to give a small p-value. Statistical significance is not biological
importance; the fold-change threshold protects you here.

### B2.6 · Visualize: volcano and MA plots  *(4 min)*

In [ ]:
# Step B2.6a · volcano: grey first (draw order!), thresholds drawn, top genes labelled, counts in the legend
x = de["log2FC_shrunk"]
y = -np.log10(de["padj"])
rest = tested & ~up & ~dn
fig, ax = plt.subplots(figsize=(5.2, 4.2))
ax.scatter(x[rest], y[rest], s=4, color=dviz.GREY, rasterized=True, label=f"not changed ({rest.sum():,})")
ax.scatter(x[up], y[up], s=9, color=dviz.VERMILION, label=f"up ({up.sum()})")
ax.scatter(x[dn], y[dn], s=9, color=dviz.OI_BLUE, label=f"down ({dn.sum()})")
ax.axhline(-np.log10(0.05), color=dviz.SLATE, ls="--", lw=0.8)
for v in (-1, 1):
    ax.axvline(v, color=dviz.SLATE, ls="--", lw=0.8)
lim = np.ceil(np.nanmax(np.abs(x[tested])))
ax.set_xlim(-lim, lim)
top8 = de[tested].nsmallest(8, "padj")
texts = [ax.text(r["log2FC_shrunk"], -np.log10(r["padj"]), r["gene"], fontsize=7, style="italic") for _, r in top8.iterrows()]
try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax, expand=(1.2, 1.6), arrowprops=dict(arrowstyle="-", color=dviz.SLATE, lw=0.5))
except ImportError:
    pass
ax.set_xlabel("Shrunken log2 fold change (treated vs control)")
ax.set_ylabel("−log10 adjusted p (BH)")
ax.set_title(f"FDR < 0.05 and |log2FC| > 1 · n = 6 vs 6")
ax.legend(fontsize=7, loc="upper center", markerscale=1.5, frameon=True, facecolor="white", edgecolor="none", framealpha=1)
fig.text(0.01, 0.005, f"{(~tested).sum()} genes not tested (padj NA) are not shown. DESeq2, design ~ batch + condition.",
         fontsize=6.5, color=dviz.SLATE)
fig.tight_layout(rect=(0, 0.03, 1, 1))
dviz.savefig(fig, "B2_volcano")
plt.show()

In [ ]:
# Step B2.6b · MA plot: raw vs shrunken fold changes against mean expression
A = np.log2(de["baseMean"] + 1)
sig = up | dn
fig, axs = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, col, t in [(axs[0], "log2FC", "Raw log2 fold change"), (axs[1], "log2FC_shrunk", "Shrunken log2 fold change")]:
    ax.scatter(A[tested & ~sig], de.loc[tested & ~sig, col], s=3, color=dviz.GREY, rasterized=True)
    ax.scatter(A[sig], de.loc[sig, col], s=5, color=dviz.BLUE)
    ax.axhline(0, color=dviz.SLATE, lw=0.8)
    ax.set_title(t)
    ax.set_xlabel("A = log2 mean normalised count")
axs[0].set_ylabel("M = log2 fold change")
axs[0].set_ylim(-6, 6)
fig.tight_layout()
dviz.savefig(fig, "B2_ma_plot")
plt.show()

In [ ]:
# CHECK · the labelled genes, and the known glucocorticoid-response genes among the "up" genes
known = truth.index[truth["known_gr_target"] == 1]
dviz.check("B2.top8_genes", ", ".join(top8["gene"]))
dviz.check("B2.known_gr_up", int(de.loc[up, "gene"].isin(known).sum()), between=(12, len(known)), note=f"of {len(known)} known targets")
low_a = tested & (A < 4)
dviz.check("B2.shrinkage_low_counts", round(float(de.loc[low_a, "log2FC"].abs().median() / max(de.loc[low_a, "log2FC_shrunk"].abs().median(), 1e-6)), 1),
           note="raw / shrunken |log2FC| for genes with A < 4")

**What changed?** The volcano now carries the evidence a reader needs to check the claim: adjusted p-values, shrunken
fold changes, both thresholds drawn, colour-blind-safe colours, the counts, and a note about the untested genes.
The most significant genes include the planted glucocorticoid-response genes (for example *FKBP5* and *TSC22D3*).
In the MA plot, raw fold changes scatter widely at low counts (left), and shrinkage pulls those uncertain genes
toward 0 (right).

**Your turn.** Replace `padj` by `pvalue` on the y-axis and in the threshold. How many genes would you call up now?
Are the extra genes planted?

<details><summary>Answer</summary>

More genes pass, and most of the extra ones are not planted. The y-axis must use the adjusted p-value when the
threshold is an FDR.
</details>

### B2.7 · Heatmap of the top 40 genes  *(optional, 3 min)*

**Why this step.** A heatmap shows a pattern across genes and samples at once. Three choices must be stated:
scaling (z-score each gene, so the colour shows the pattern rather than the expression level), the colour map
(diverging, centred at 0, with a colour bar), and clustering (distance and linkage). Annotation bars show whether the
clusters follow treatment or batch.

In [ ]:
# Step B2.7 · z-scored rows, correlation distance, average linkage, annotation bars
from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
top40 = de[tested].nsmallest(40, "padj")["gene"]
Z = log2n.loc[top40]
Z = Z.sub(Z.mean(axis=1), axis=0).div(Z.std(axis=1), axis=0)          # z-score each gene (row)
r_order = leaves_list(linkage(Z.to_numpy(), "average", metric="correlation"))
c_order = leaves_list(linkage(Z.T.to_numpy(), "average", metric="correlation"))
Zo = Z.iloc[r_order, c_order]
bwo = LinearSegmentedColormap.from_list("bwo", [dviz.DARKBLUE, dviz.SKY, "white", "#F2C185", dviz.VERMILION])
fig = plt.figure(figsize=(7, 6.5))
gs = fig.add_gridspec(3, 2, height_ratios=[0.25, 0.25, 9], width_ratios=[20, 0.6], hspace=0.05, wspace=0.05)
m2 = meta.set_index("sample").loc[Zo.columns]
for k, (col, cmap) in enumerate([("condition", {"control": dviz.SLATE, "treated": dviz.BLUE}),
                                 ("batch", {"A": dviz.GREY, "B": dviz.ORANGE})]):
    a = fig.add_subplot(gs[k, 0])
    a.imshow([[matplotlib.colors.to_rgb(cmap[v]) for v in m2[col]]], aspect="auto")
    a.set_yticks([0], [col], fontsize=7)
    a.set_xticks([])
    for sp in a.spines.values():
        sp.set_visible(False)
ax = fig.add_subplot(gs[2, 0])
im = ax.imshow(Zo, aspect="auto", cmap=bwo, norm=TwoSlopeNorm(0, -2.5, 2.5), interpolation="nearest")
ax.set_yticks(range(len(Zo)), Zo.index, fontsize=6, style="italic")
ax.set_xticks(range(Zo.shape[1]), Zo.columns, rotation=90, fontsize=7)
fig.colorbar(im, cax=fig.add_subplot(gs[2, 1]), label="row z-score")
fig.suptitle("Top 40 DE genes · z-scored rows · 1 − Pearson r, average linkage", x=0.08, ha="left", fontsize=9, weight="bold")
dviz.savefig(fig, "B2_heatmap")
plt.show()

In [ ]:
# CHECK · rows are scaled; the columns split first by treatment
dviz.check("B2.heatmap_row_mean_max", round(float(Z.mean(axis=1).abs().max()), 6), 0.0, tol=1e-6)
dviz.check("B2.heatmap_row_sd", round(float(Z.std(axis=1).mean()), 3), 1.0, tol=0.001)
first_half = m2["condition"].iloc[:6]
dviz.check("B2.heatmap_columns_split_by_condition", bool(first_half.nunique() == 1), True)

**What changed?** The 40 genes form two blocks, up and down with treatment, and the column dendrogram separates
treated from control first. The batch annotation bar is mixed within each block: for these genes, treatment
dominates.

### B2 · Summary

| Step | What you did | Number that proves it |
|---|---|---|
| B2.1 | looked at counts and design | 8,000 genes × 12 samples, 3 + 3 per batch |
| B2.2 | median-of-ratios normalisation | size factors 0.65–1.47 |
| B2.3 | PCA for QC | PC1 ≈ treatment (39%), PC2 ≈ batch (23%) |
| B2.4 | 6,940 tests, BH | 3.9% of called genes not planted; batch in the model finds 9 more real changes |
| B2.5 | DESeq2 results | 7,392 tested, 608 not tested; 177 up, 106 down |
| B2.6 | volcano + MA | thresholds drawn, counts in the legend |

**Presentation sentence (example).** *"Which genes respond to dexamethasone? 177 genes go up and 106 go down at
FDR 5% and |log2FC| > 1 (DESeq2, 6 vs 6, batch in the model). Limitation: a batch effect explains 23% of the
variation; the design is balanced, so we could model it. The AI wrote the first plotting code; we checked every count
against the notebook."*

# Defend and present  *(10 min)*

## Step D1 · Verify: do the figure, the legend and the numbers agree?

Every number in a figure or a legend must come from printed output. The cell below lists all CHECK values from this
notebook. Compare them with your figure legends and with the reference table on the slide.

In [ ]:
# Step D1 · all CHECK values of this notebook
res = pd.DataFrame(dviz._RESULTS).T[["value", "reference", "ok"]]
print(f"{(res['ok'] == True).sum()} passed, {(res['ok'] == False).sum()} failed, {res['ok'].isna().sum()} recorded only")
res

## Step D2 · Accessibility: colour vision and greyscale

About 1 in 12 men (and 1 in 200 women) of Northern European ancestry have red–green colour-vision deficiency.
`dviz.show_cvd()` shows a saved figure as it looks with deuteranopia (no green-sensitive cones) and in greyscale.
Try it on your volcano and on the misleading volcano M6.

In [ ]:
# Step D2 · simulate deuteranopia and greyscale for your volcano and for M6
for path in [FIG / "B2_volcano.png", ROOT / "bad_figures" / "M6.png"]:
    if path.exists():
        f = dviz.show_cvd(path, width=10)
        f.suptitle(path.name, x=0.01, ha="left", fontsize=9)
        plt.show()

**What to look for.** In M6 the red and green groups become the same olive tone, so up and down can no longer be told
apart. In your volcano, vermilion and blue stay distinct, and the threshold lines and counts still carry the message.
If two groups look the same, switch to the Okabe–Ito colours **and** add a second channel (shape or direct labels).

## Step D3 · Words, AI and the log  *(optional)*

1. Ask an AI assistant for **alt text (≤ 60 words)** and a **figure legend** using only your code and the printed
   numbers (not the image). Correct anything it could not know (biology, sample origin) and anything that does not
   match the printed numbers.
2. Record what you did in `ai_use_log.md`: tool, date, prompt, what you accepted, what you changed, how you checked.

## Present one figure (2 minutes per pair)

**Question** (one sentence) → **Figure** (explain the axes first) → **Takeaway** with a number → **Limitation** →
**AI and checks** (what the AI did and how you verified it). Use the review checklist in `checklist.md`.

In [ ]:
# Save all CHECK values (used by the instructor) and the run time
for ex in ["S0", "A", "B1", "B2"]:
    dviz.save_results(ex)
print(f"Notebook time: {time.time() - T_START:.0f} s")